# Composable Modes

Production AI agents rarely operate in a single, flat configuration. A legal research assistant needs to combine a specific level of human oversight with a research-oriented reasoning style and legal domain constraints - all at the same time. A CI/CD security reviewer needs fully autonomous execution, code-analysis reasoning, and security-specific tool restrictions. Writing a separate monolithic system prompt for every possible combination of oversight level, reasoning style, and domain quickly becomes unmaintainable - and worse, inconsistent.

Mode composition solves this by separating an agent's configuration into three independent, stackable layers, each controlling a distinct dimension of behavior. We define each layer once and combine them at runtime into a complete, coherent configuration. This pattern appears in production AI platforms including Google Vertex AI agent configs, Microsoft AutoGen's agent profiles, and enterprise deployments that need to serve many use cases from a shared agent infrastructure.

In [1]:
import os
from dataclasses import dataclass, field

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated, Sequence
from langchain_core.messages import BaseMessage
from langgraph.graph.message import add_messages

### Initialize the language model

A low temperature setting produces consistent, deterministic output. This matters here because we want to observe the effect of different mode configurations - not random variation - when we compare agent responses side by side later in the notebook.

In [2]:
llm = ChatOpenAI(model="gpt-4o-mini", api_key=os.getenv("OPENAI_API_KEY", "").strip(), temperature=0.3)
print("LLM ready:", llm.model_name)

LLM ready: gpt-4o-mini


## The three-layer composition model
An agent's full configuration can always be decomposed into three orthogonal dimensions. "Orthogonal" means each dimension controls something completely independent - changing the autonomy level does not change the reasoning style, and adding a legal domain constraint does not change whether approval gates are required. This independence is what makes composition practical: we can swap any one layer without touching the others.

```
FULL MODE = AUTONOMY LAYER + BEHAVIORAL LAYER + DOMAIN LAYER

┌────────────────────────────────────────────────────────────────┐
│  AUTONOMY LAYER   ── How much human oversight is in the loop   │
│  (required)          exception_only | approval_gated |         │
│                      fully_automated                           │
│                                                                │
│  BEHAVIORAL LAYER ── How the agent reasons and communicates    │
│  (required)          research | code | advisory | task         │
│                                                                │
│  DOMAIN LAYER     ── What the agent knows and is bound by      │
│  (optional)          legal | compliance | security             │
└────────────────────────────────────────────────────────────────┘
```

The composition order is always **Autonomy → Behavioral → Domain**. Domain layers apply last because they can only tighten - never loosen - what earlier layers established: a security domain can remove tools from the behavioral whitelist or raise the escalation threshold, but it cannot grant autonomous execution to an approval-gated agent.

| Layer      | Controls                                                       |
|------------|----------------------------------------------------------------|
| Autonomy   | Approval gates, escalation thresholds, human review depth      |
| Behavioral | Reasoning style, output format, default tool set               |
| Domain     | Topic restrictions, regulatory rules, tool access tightening   |

Exactly one autonomy layer and one behavioral layer are required per composed configuration. Domain layers are optional, but they are the norm in any regulated or sensitive production context.

## Defining a mode layer
Each layer is a small data structure that carries two outputs: a `config` dictionary consumed by infrastructure code (tool routing, graph construction, escalation logic), and a `system_prompt_fragment` consumed by the LLM. Keeping both in the same structure ensures they stay in sync - if we update a layer's behavior, we update both outputs together.

In [3]:
@dataclass
class ModeLayer:
    """A single composable configuration layer for an AI agent mode.

    Attributes:
        layer_type: One of 'autonomy', 'behavioral', or 'domain'.
        layer_name: The specific mode within that type (e.g. 'research', 'legal').
        config: Structured settings consumed by infrastructure code at runtime.
        system_prompt_fragment: Natural language instructions contributed to the LLM system prompt.
    """
    layer_type: str
    layer_name: str
    config: dict = field(default_factory=dict)
    system_prompt_fragment: str = ""


# Show what a minimal layer looks like in practice
example = ModeLayer(
    layer_type="autonomy",
    layer_name="example",
    config={"approval_required": False, "escalate_on_confidence_below": 0.75},
    system_prompt_fragment="Operate autonomously unless confidence drops below 75%.",
)
print(f"type     : {example.layer_type}")
print(f"name     : {example.layer_name}")
print(f"config   : {example.config}")
print(f"fragment : {example.system_prompt_fragment}")

type     : autonomy
name     : example
config   : {'approval_required': False, 'escalate_on_confidence_below': 0.75}
fragment : Operate autonomously unless confidence drops below 75%.


`ModeLayer` is pure data - it carries no logic. The `config` dict holds values that infrastructure code checks at runtime to make routing and control flow decisions. The `system_prompt_fragment` is a plain string that gets concatenated with fragments from other layers to form the final system prompt. The two outputs serve entirely different consumers: config is for code, fragments are for the model.

## Autonomy layers
The autonomy layer answers a single question: how much human oversight does this agent configuration require? The three layers below cover the spectrum from exception-only oversight (the most common production default) through approval-gated (human in the loop for all consequential actions) to fully automated (no human involvement at all). Notice that `fully_automated` pairs its zero-oversight setting with `audit_depth: full` - when no human reviews execution in real time, the audit trail becomes the primary accountability mechanism.

In [4]:
# --- exception_only: autonomous by default, escalate only on errors or low confidence ---
exception_only = ModeLayer(
    layer_type="autonomy",
    layer_name="exception_only",
    config={
        "autonomy_level": "exception_only",
        "approval_required": False,           # No pre-action approval needed
        "escalate_on_confidence_below": 0.75,  # Stop and escalate when uncertain
        "human_review": "on_exception",        # Review only when something goes wrong
        "audit_depth": "standard",             # Log key decisions
    },
    system_prompt_fragment=(
        "Operate autonomously. Escalate to a human reviewer only when your confidence "
        "is below 75% or when you encounter an unrecoverable error. "
        "Log your reasoning at each key decision for audit purposes."
    ),
)

# --- approval_gated: pause and request human sign-off before any external or write action ---
approval_gated = ModeLayer(
    layer_type="autonomy",
    layer_name="approval_gated",
    config={
        "autonomy_level": "approval_gated",
        "approval_required": True,             # Require explicit approval before acting
        "escalate_on_confidence_below": 0.90,  # Higher bar — stop more conservatively
        "approval_gates": ["before_external_comms", "before_writes"],  # Where gates trigger
        "human_review": "before_action",       # Review before each consequential step
        "audit_depth": "full",                 # Full audit trail for every decision
    },
    system_prompt_fragment=(
        "Always seek explicit human approval before sending any external communications "
        "or performing any write operations. Present your proposed action clearly "
        "and wait for confirmation before proceeding."
    ),
)

# --- fully_automated: no human in the loop; relies on audit logging for oversight ---
fully_automated = ModeLayer(
    layer_type="autonomy",
    layer_name="fully_automated",
    config={
        "autonomy_level": "fully_automated",
        "approval_required": False,            # Fully hands-off — no approvals
        "escalate_on_confidence_below": 0.50,  # Only stop for very low confidence
        "human_review": "none",                # No human review during execution
        "audit_depth": "full",                 # Full audit compensates for absent HITL
    },
    system_prompt_fragment=(
        "Operate fully autonomously without requiring human approval. "
        "Make all decisions independently and document your full reasoning "
        "for compliance audit."
    ),
)

print(f"{'Layer':<20} {'Approval':>10}  {'Escalate at':>12}  Human review")
print("-" * 60)
for layer in [exception_only, approval_gated, fully_automated]:
    gate = "yes" if layer.config["approval_required"] else "no"
    threshold = layer.config["escalate_on_confidence_below"]
    review = layer.config["human_review"]
    print(f"{layer.layer_name:<20} {gate:>10}  {threshold:>12}  {review}")

Layer                  Approval   Escalate at  Human review
------------------------------------------------------------
exception_only               no          0.75  on_exception
approval_gated              yes           0.9  before_action
fully_automated              no           0.5  none


The escalation threshold rises as autonomy decreases: `fully_automated` proceeds unless confidence drops below 50%, while `approval_gated` stops at 90%. This relationship is intentional - lower autonomy means more conservative stops, not fewer. The `audit_depth` field also illustrates the safety compensation pattern: when `human_review` is `none`, audit is always set to `full`.

## Behavioral layers
The behavioral layer answers a different question: how does this agent think and communicate? It controls reasoning style, output format, and the default set of tools appropriate for that mode. The same autonomy level can be paired with research reasoning or code-analysis reasoning - these are genuinely independent choices. Notice that `code_behavioral` includes `web_search` in its tool set (useful for looking up documentation), while the security domain layer will later remove it - this is a concrete example of domain tightening at the tool level.

In [5]:
# --- research: gather from multiple sources, synthesize, cite evidence ---
research_behavioral = ModeLayer(
    layer_type="behavioral",
    layer_name="research",
    config={
        "reasoning_style": "multi-source",        # Draw from multiple information sources
        "planning_depth": "deep",                  # Think through the approach before acting
        "output_format": "research_report",        # Structured report with citations
        "sources_required": 3,                     # Minimum sources before concluding
        "tool_whitelist": ["web_search", "academic_db", "document_read", "summarizer"],
    },
    system_prompt_fragment=(
        "Use a research-oriented reasoning style: gather from multiple sources, "
        "evaluate credibility, synthesize findings into a coherent conclusion, "
        "and cite your sources. Clearly indicate the strength of evidence behind each claim."
    ),
)

# --- code: analytical, structured, review-before-execute ---
code_behavioral = ModeLayer(
    layer_type="behavioral",
    layer_name="code",
    config={
        "reasoning_style": "analytical",           # Step-by-step structured analysis
        "planning_depth": "structured",            # Plan then implement
        "output_format": "code_with_explanation",  # Code blocks with inline commentary
        "review_before_execute": True,             # Review logic before suggesting execution
        # web_search is included so developers can look up docs — note it will be
        # removed when the security domain layer is added
        "tool_whitelist": ["code_interpreter", "file_read", "test_runner", "linter", "web_search"],
    },
    system_prompt_fragment=(
        "Approach tasks analytically: understand requirements fully before writing code, "
        "plan the implementation, then write clean and commented code. "
        "Always review logic before suggesting execution. Propose tests alongside implementations."
    ),
)

print(f"{'Layer':<12}  {'Reasoning style':<16}  Tools")
print("-" * 65)
for layer in [research_behavioral, code_behavioral]:
    style = layer.config["reasoning_style"]
    tools = layer.config["tool_whitelist"]
    print(f"{layer.layer_name:<12}  {style:<16}  {tools}")

Layer         Reasoning style   Tools
-----------------------------------------------------------------
research      multi-source      ['web_search', 'academic_db', 'document_read', 'summarizer']
code          analytical        ['code_interpreter', 'file_read', 'test_runner', 'linter', 'web_search']


The behavioral layer is the primary driver of the LLM's reasoning pattern. `research` instructs the model to work multi-source and synthesize with citations. `code` instructs it to plan then implement, with a mandatory review step. The `tool_whitelist` in each behavioral layer defines which tools are appropriate for that reasoning mode as a baseline - domain layers may then restrict or extend this set.

## Domain layers
Domain layers are different from the other two in an important way: they can only make the configuration more restrictive, not more permissive. A domain layer can raise the escalation threshold (but never lower it), add domain-specific tools the behavioral layer didn't include (like a statute lookup database for legal work), and block tools that are inappropriate for the domain (outbound HTTP calls in a security reviewer). It cannot remove approval gates that the autonomy layer established. This one-directional constraint is what makes domain layers safe to stack onto any combination of autonomy and behavioral layers.

In [6]:
# --- legal: formal tone, primary source citations, no legal advice ---
legal_domain = ModeLayer(
    layer_type="domain",
    layer_name="legal",
    config={
        # domain_tools are added to the behavioral whitelist (additive)
        "domain_tools": ["legal_search", "statute_lookup", "case_law_db", "document_read"],
        "can_make_recommendations": False,          # Information only — no legal advice
        "cite_primary_sources": True,               # Must cite statutes and case law
        "output_disclaimer": "This is not legal advice. Consult a qualified attorney.",
        "escalate_on_confidence_below": 0.85,       # Stricter than most autonomy defaults
        "tone": "formal",                           # Formal and precise language
    },
    system_prompt_fragment=(
        "You operate in a legal research context. Provide legal information only — "
        "never legal advice. Always cite primary sources (statutes, case law, regulations). "
        "Note jurisdictional differences where relevant. Maintain formal, precise language. "
        "Append to all responses: 'This is not legal advice. Consult a qualified attorney.'"
    ),
)

# --- compliance: regulatory citations, jurisdiction flags, mandatory disclaimers ---
compliance_domain = ModeLayer(
    layer_type="domain",
    layer_name="compliance",
    config={
        "domain_tools": ["regulatory_db", "statute_lookup", "document_read"],
        "require_regulatory_citations": True,       # Every claim must cite a regulation
        "flag_jurisdiction_specific": True,         # Explicitly flag country/region variations
        "output_disclaimer_required": True,         # All output needs a disclaimer
        "escalate_on_violation_detected": True,     # Immediately escalate on detected violation
        "audit_depth": "full",                      # Compliance always requires a full audit trail
    },
    system_prompt_fragment=(
        "You operate in a compliance context. Always cite applicable regulations "
        "(GDPR, HIPAA, SOX, etc.) when discussing requirements. "
        "Flag jurisdiction-specific variations explicitly. Never present compliance "
        "information as legal advice. Append: 'Consult qualified legal counsel for specific guidance.'"
    ),
)

# --- security: OWASP-focused, no outbound network, structured severity output ---
security_domain = ModeLayer(
    layer_type="domain",
    layer_name="security",
    config={
        "domain_tools": ["sast_scanner", "dependency_audit", "file_read", "code_interpreter"],
        # blocked_tools are removed from the whitelist — outbound network calls are a
        # security risk in a code review context
        "blocked_tools": ["web_search", "external_http"],
        "execute_untrusted_code": False,            # Never run unreviewed code
        "escalate_on_critical_vulnerability": True, # Critical findings always escalate
        "escalate_on_confidence_below": 0.80,       # Raise the floor for security assessments
        "output_format": "security_report",         # Structured severity ratings in output
    },
    system_prompt_fragment=(
        "You operate in a security review context. Check for OWASP Top 10 vulnerabilities, "
        "insecure dependencies, and hardcoded secrets. Never execute untrusted code. "
        "Flag critical findings for immediate escalation. "
        "Structure output with severity ratings: Critical / High / Medium / Low."
    ),
)

print(f"{'Domain':<12}  {'Escalate at':>12}  Domain tools added")
print("-" * 65)
for layer in [legal_domain, compliance_domain, security_domain]:
    threshold = layer.config.get("escalate_on_confidence_below", "— (inherits)")
    tools = layer.config.get("domain_tools", [])
    print(f"{layer.layer_name:<12}  {str(threshold):>12}  {tools}")

Domain         Escalate at  Domain tools added
-----------------------------------------------------------------
legal                 0.85  ['legal_search', 'statute_lookup', 'case_law_db', 'document_read']
compliance    — (inherits)  ['regulatory_db', 'statute_lookup', 'document_read']
security               0.8  ['sast_scanner', 'dependency_audit', 'file_read', 'code_interpreter']


Domain layers use a `domain_tools` key (distinct from `tool_whitelist`) to make the semantics explicit - these are tools the domain makes available in addition to the behavioral layer's baseline. The composition step merges them. The `blocked_tools` key is a deny list: any tool listed there is removed from the whitelist during composition, regardless of what the behavioral layer included. In the security domain, `web_search` is blocked specifically because outbound network calls from a code reviewer would themselves be a vulnerability vector.

## Composing layers into a full configuration
Before we can compose any stack, we need two utility functions: one that validates a layer set (enforcing the requirement for exactly one autonomy and one behavioral layer), and one that sorts layers into the canonical order. These are small enough to keep as standalone functions rather than wrapping them in a class - the composition logic that follows is the real substance.

In [7]:
# The canonical layer application order — domain always goes last
LAYER_ORDER = ["autonomy", "behavioral", "domain"]


def validate_layers(layers: list) -> None:
    """Raise ValueError if the layer set is missing a required type or has duplicates."""
    layer_types = [l.layer_type for l in layers]  # collect all layer types in this stack
    for required in ["autonomy", "behavioral"]:    # both types are mandatory
        count = layer_types.count(required)
        if count == 0:
            raise ValueError(f"Missing required layer type: '{required}'")
        if count > 1:
            # Two autonomy layers would create ambiguity in approval gate logic
            raise ValueError(f"Only one '{required}' layer allowed per stack, found {count}")


def _sort_layers(layers: list) -> list:
    """Sort layers into the canonical composition order: autonomy → behavioral → domain."""
    return sorted(
        layers,
        # Assign sort key by position in LAYER_ORDER; unknowns go to the end
        key=lambda l: LAYER_ORDER.index(l.layer_type) if l.layer_type in LAYER_ORDER else 99
    )


# Confirm validation catches a stack missing its autonomy layer
try:
    validate_layers([research_behavioral, compliance_domain])  # no autonomy layer
except ValueError as e:
    print(f"Validation caught: {e}")

# A valid stack passes silently
validate_layers([exception_only, research_behavioral, compliance_domain])
print("Validation passed: exception_only + research_behavioral + compliance_domain")

Validation caught: Missing required layer type: 'autonomy'
Validation passed: exception_only + research_behavioral + compliance_domain


`validate_layers` is called at the start of every compose operation, so invalid stacks fail immediately with a clear message rather than silently producing a broken configuration. The sort function is prefixed with `_` to signal that it is an implementation detail used internally by the composition functions, not part of the public interface.

## The compose and system prompt functions
The `compose_layers` function is the heart of the composition model. It iterates layers in order - autonomy first, then behavioral, then domain - applying a simple merge for the first two and an explicit tightening protocol for domain layers. The tightening protocol encodes five named rules that make the domain-layer semantics auditable: we can read exactly what happens to each kind of key when a domain layer is applied.

In [8]:
def compose_layers(layers: list) -> dict:
    """Merge mode layers into a single resolved configuration dict.

    Autonomy is the base. Behavioral adds its config on top, overriding where keys overlap.
    Domain layers apply last with explicit tightening rules — they cannot loosen anything the autonomy or behavioral layers established.
    """
    validate_layers(layers)                 # fail fast on invalid stacks
    ordered = _sort_layers(layers)          # apply in canonical order
    resolved = {}                           # accumulates the merged result

    for layer in ordered:
        if layer.layer_type == "domain":
            # Domain layers tighten — each key type has its own explicit rule
            for key, value in layer.config.items():

                if key == "domain_tools":
                    # Add domain-specific tools to the behavioral whitelist (additive)
                    existing = resolved.get("tool_whitelist", [])
                    # Append only tools not already in the whitelist to avoid duplicates
                    resolved["tool_whitelist"] = existing + [t for t in value if t not in existing]

                elif key == "blocked_tools":
                    # Union: a tool blocked by either layer stays blocked overall
                    existing_blocks = resolved.get("blocked_tools", [])
                    new_blocks = list(set(existing_blocks) | set(value))
                    resolved["blocked_tools"] = new_blocks
                    # Remove any newly blocked tools from the whitelist
                    whitelist = resolved.get("tool_whitelist", [])
                    resolved["tool_whitelist"] = [t for t in whitelist if t not in new_blocks]

                elif key == "escalate_on_confidence_below":
                    # Domain can only raise the escalation floor — take the higher threshold
                    resolved[key] = max(resolved.get(key, 0), value)

                elif key == "audit_depth" and value == "full":
                    # Audit depth can only upgrade to full, never downgrade
                    resolved[key] = "full"

                elif key == "approval_required" and value is True:
                    # Domain cannot remove an approval gate the autonomy layer established
                    resolved[key] = True

                else:
                    # All other domain keys: add to resolved only if not already present
                    # (autonomy/behavioral settings take precedence over domain defaults)
                    resolved.setdefault(key, value)
        else:
            # Autonomy (first) and behavioral (second): straight merge
            # Later values override earlier ones — behavioral overrides autonomy defaults
            resolved.update(layer.config)

    # Record which layers contributed — useful for logging and debugging
    resolved["_layers"] = [f"{l.layer_type}:{l.layer_name}" for l in ordered]
    return resolved


def build_system_prompt(layers: list) -> str:
    """Concatenate layer prompt fragments in composition order."""
    ordered = _sort_layers(layers)
    # Collect non-empty fragments — some layers may have no prompt contribution
    fragments = [l.system_prompt_fragment for l in ordered if l.system_prompt_fragment.strip()]
    return "\n\n".join(fragments)  # blank line between fragments keeps the prompt readable


print("compose_layers and build_system_prompt ready.")

compose_layers and build_system_prompt ready.


Five explicit rules handle the five domain tightening cases: `domain_tools` are additive (merged into the whitelist), `blocked_tools` grow by union, the escalation threshold takes the maximum (floor rises), `audit_depth` can only upgrade to full, and `approval_required` uses OR logic (domain cannot remove a gate). Every other domain key uses `setdefault` - the autonomy or behavioral layer's value wins when there is a conflict, because the domain layer is adding context-specific metadata, not overriding the core control settings.

The `build_system_prompt` function is intentionally simple: sort, filter empty, join with blank lines. The order - autonomy instructions first, behavioral reasoning style second, domain constraints last - ensures the LLM receives the most general operating instructions before the most specific constraints.

## Composing mode stacks
Let's build two complete stacks and inspect their resolved configurations. The first pairs autonomous exception-only oversight with research reasoning in a compliance context - the kind of stack a regulatory monitoring agent would use. The second pairs fully automated execution with code analysis in a security context - the kind of stack a CI/CD pipeline reviewer would use.

In [9]:
# Stack A: Autonomous Compliance Research
# Use case: agent that monitors new regulations and synthesizes compliance implications
compliance_research_layers = [exception_only, research_behavioral, compliance_domain]
compliance_cfg = compose_layers(compliance_research_layers)

print("=== Stack A: Autonomous Compliance Research ===")
print(f"Layers       : {compliance_cfg['_layers']}")
print(f"Approval req : {compliance_cfg.get('approval_required')}")
print(f"Escalate at  : {compliance_cfg.get('escalate_on_confidence_below')}")
print(f"Audit depth  : {compliance_cfg.get('audit_depth')}")
print(f"Tool access  : {compliance_cfg.get('tool_whitelist')}")
print(f"Reg. cite    : {compliance_cfg.get('require_regulatory_citations')}")
print()

# Stack B: Automated Security Code Review
# Use case: CI/CD agent that reviews every pull request for security vulnerabilities
security_code_layers = [fully_automated, code_behavioral, security_domain]
security_cfg = compose_layers(security_code_layers)

print("=== Stack B: Automated Security Code Review ===")
print(f"Layers       : {security_cfg['_layers']}")
print(f"Approval req : {security_cfg.get('approval_required')}")
print(f"Escalate at  : {security_cfg.get('escalate_on_confidence_below')}")
print(f"Audit depth  : {security_cfg.get('audit_depth')}")
print(f"Tool access  : {security_cfg.get('tool_whitelist')}")
print(f"Blocked      : {security_cfg.get('blocked_tools')}")
print(f"Output fmt   : {security_cfg.get('output_format')}")

=== Stack A: Autonomous Compliance Research ===
Layers       : ['autonomy:exception_only', 'behavioral:research', 'domain:compliance']
Approval req : False
Escalate at  : 0.75
Audit depth  : full
Tool access  : ['web_search', 'academic_db', 'document_read', 'summarizer', 'regulatory_db', 'statute_lookup']
Reg. cite    : True

=== Stack B: Automated Security Code Review ===
Layers       : ['autonomy:fully_automated', 'behavioral:code', 'domain:security']
Approval req : False
Escalate at  : 0.8
Audit depth  : full
Tool access  : ['code_interpreter', 'file_read', 'test_runner', 'linter', 'sast_scanner', 'dependency_audit']
Blocked      : ['web_search', 'external_http']
Output fmt   : code_with_explanation


Stack A inherits the `audit_depth: full` from the compliance domain, which upgraded from the `standard` setting in `exception_only` - a domain tightening in action. Stack B shows the security domain raising the escalation threshold from 0.50 (`fully_automated`) to 0.80 (the security domain's floor), and the tool list reflects both additions (`sast_scanner`, `dependency_audit`) and the removal of `web_search` via `blocked_tools`.

## Generating system prompts from composed stacks
Each layer contributes one fragment to the final system prompt. The composed prompt is the primary mechanism by which mode configuration actually changes LLM behavior - infrastructure code reads the `config` dict, but the model reads the system prompt. Let's see what the prompts for each stack look like.

In [10]:
compliance_prompt = build_system_prompt(compliance_research_layers)
security_prompt = build_system_prompt(security_code_layers)

print("=== System Prompt: Compliance Research Stack ===\n")
print(compliance_prompt)
print()
print("=" * 60)
print()
print("=== System Prompt: Security Code Review Stack ===\n")
print(security_prompt)

=== System Prompt: Compliance Research Stack ===

Operate autonomously. Escalate to a human reviewer only when your confidence is below 75% or when you encounter an unrecoverable error. Log your reasoning at each key decision for audit purposes.

Use a research-oriented reasoning style: gather from multiple sources, evaluate credibility, synthesize findings into a coherent conclusion, and cite your sources. Clearly indicate the strength of evidence behind each claim.

You operate in a compliance context. Always cite applicable regulations (GDPR, HIPAA, SOX, etc.) when discussing requirements. Flag jurisdiction-specific variations explicitly. Never present compliance information as legal advice. Append: 'Consult qualified legal counsel for specific guidance.'


=== System Prompt: Security Code Review Stack ===

Operate fully autonomously without requiring human approval. Make all decisions independently and document your full reasoning for compliance audit.

Approach tasks analytically:

Each composed prompt consists of three fragments joined in order: autonomy instructions (how to handle escalation and logging), behavioral instructions (how to reason and format output), and domain constraints (what to check and what to append to responses). A reader of the final prompt cannot see the layer boundaries - which is by design. The LLM receives a coherent set of instructions, not a visible composition artifact.

## How the security domain tightens tool access
The most concrete illustration of domain tightening is what happens to the tool whitelist. The `code_behavioral` layer includes `web_search` for documentation lookups - a reasonable default for code work. The `security_domain` blocks it because outbound network calls from a code reviewer are a potential exfiltration path. Let's trace exactly what changes when the security domain is added.

In [11]:
# Baseline: code behavioral layer without any domain restriction
base_cfg = compose_layers([exception_only, code_behavioral])
print("code_behavioral (no domain):")
print(f"  Whitelist : {base_cfg.get('tool_whitelist')}")
print(f"  Blocked   : {base_cfg.get('blocked_tools', [])}")
print(f"  Escalate  : {base_cfg.get('escalate_on_confidence_below')}")
print()

# With security domain: tools are added, blocked, and escalation threshold rises
security_code_cfg = compose_layers([exception_only, code_behavioral, security_domain])
print("code_behavioral + security_domain:")
print(f"  Whitelist : {security_code_cfg.get('tool_whitelist')}")
print(f"  Blocked   : {security_code_cfg.get('blocked_tools')}")
print(f"  Escalate  : {security_code_cfg.get('escalate_on_confidence_below')}")
print()

# Summarise what the domain layer actually changed
base_tools = set(base_cfg.get("tool_whitelist", []))
sec_tools = set(security_code_cfg.get("tool_whitelist", []))

added = sec_tools - base_tools     # tools the security domain contributed
removed = base_tools - sec_tools   # tools the security domain blocked and removed

print(f"  Added by security domain   : {sorted(added)}")
print(f"  Removed by security domain : {sorted(removed)}")
print(f"  Escalation threshold raised: 0.75 → {security_code_cfg.get('escalate_on_confidence_below')}")

code_behavioral (no domain):
  Whitelist : ['code_interpreter', 'file_read', 'test_runner', 'linter', 'web_search']
  Blocked   : []
  Escalate  : 0.75

code_behavioral + security_domain:
  Whitelist : ['code_interpreter', 'file_read', 'test_runner', 'linter', 'sast_scanner', 'dependency_audit']
  Blocked   : ['web_search', 'external_http']
  Escalate  : 0.8

  Added by security domain   : ['dependency_audit', 'sast_scanner']
  Removed by security domain : ['web_search']
  Escalation threshold raised: 0.75 → 0.8


`sast_scanner` and `dependency_audit` were added - these are domain-specific tools that `code_behavioral` didn't include because they are only relevant in a security context. `web_search` was removed because it appeared in `security_domain`'s `blocked_tools` list, which caused it to be stripped from the whitelist during composition. The escalation threshold also rose from 0.75 to 0.80 because the security domain's floor is higher than the `exception_only` autonomy layer's default. All three of these are tightening operations: the domain made the configuration more restrictive in every dimension it touched.

## Live agent demonstrations
Now let's observe how different composition stacks produce concretely different agent outputs when given questions tailored to their respective roles. The composed system prompt is the only variable - the same underlying model handles both queries. This is the most direct way to see that composable modes are not just an architectural pattern but a practical mechanism for steering model behavior.

In [12]:
def run_agent(layers: list, query: str, label: str) -> str:
    """Run a single-turn agent with the composed mode layers and print the response."""
    system_prompt = build_system_prompt(layers)
    # System message carries the composed prompt; human message carries the query
    messages = [SystemMessage(content=system_prompt), HumanMessage(content=query)]
    response = llm.invoke(messages)

    print(f"\n{'=' * 60}")
    print(f"STACK : {label}")
    print(f"QUERY : {query[:80]}{'...' if len(query) > 80 else ''}")
    print(f"{'=' * 60}")
    print(response.content)
    return response.content


# Demo 1: Compliance research stack — a regulatory question with jurisdictional nuance
run_agent(
    layers=compliance_research_layers,
    query=(
        "What are the data retention requirements for employee records under GDPR, "
        "and how do they differ for companies operating in Germany versus Ireland?"
    ),
    label="Compliance Research (exception_only + research + compliance)",
)


STACK : Compliance Research (exception_only + research + compliance)
QUERY : What are the data retention requirements for employee records under GDPR, and ho...
To address the data retention requirements for employee records under the General Data Protection Regulation (GDPR), I will gather information from multiple credible sources, evaluate their relevance, and synthesize the findings.

### General GDPR Data Retention Principles

1. **Data Minimization and Purpose Limitation**: Under Article 5(1)(c) of the GDPR, personal data must be adequate, relevant, and limited to what is necessary for the purposes for which they are processed. This implies that employee records should only be retained as long as necessary to fulfill the purpose for which they were collected.

2. **Retention Period**: There is no specific retention period mandated by the GDPR itself; rather, it requires organizations to establish their own retention policies based on the purpose of data processing and applicable

'To address the data retention requirements for employee records under the General Data Protection Regulation (GDPR), I will gather information from multiple credible sources, evaluate their relevance, and synthesize the findings.\n\n### General GDPR Data Retention Principles\n\n1. **Data Minimization and Purpose Limitation**: Under Article 5(1)(c) of the GDPR, personal data must be adequate, relevant, and limited to what is necessary for the purposes for which they are processed. This implies that employee records should only be retained as long as necessary to fulfill the purpose for which they were collected.\n\n2. **Retention Period**: There is no specific retention period mandated by the GDPR itself; rather, it requires organizations to establish their own retention policies based on the purpose of data processing and applicable laws. Organizations must consider factors such as:\n   - Legal obligations (e.g., tax laws, labor laws)\n   - The necessity of the data for the fulfillmen

In [13]:
# Demo 2: Security code review stack — a vulnerable code snippet for review
run_agent(
    layers=security_code_layers,
    query="""\
Review this Python function for security issues:

def get_user(user_id: str):
    query = f"SELECT * FROM users WHERE id = {user_id}"
    return db.execute(query)
""",
    label="Security Code Review (fully_automated + code + security)",
)


STACK : Security Code Review (fully_automated + code + security)
QUERY : Review this Python function for security issues:

def get_user(user_id: str):
  ...
### Security Review of the `get_user` Function

#### Code Analysis
The provided function retrieves a user from a database based on the `user_id` parameter. The function constructs a SQL query using string interpolation, which can lead to serious security vulnerabilities.

#### Identified Vulnerabilities
1. **SQL Injection (Critical)**: The use of string interpolation to construct the SQL query makes it vulnerable to SQL injection attacks. An attacker could manipulate the `user_id` input to execute arbitrary SQL commands.

2. **Insecure Dependencies (Medium)**: The function relies on the `db.execute()` method, but without knowing the implementation details of the `db` object, we cannot ascertain if it uses prepared statements or parameterized queries internally. If it does not, the risk of SQL injection remains.

3. **Hardcoded Sec

'### Security Review of the `get_user` Function\n\n#### Code Analysis\nThe provided function retrieves a user from a database based on the `user_id` parameter. The function constructs a SQL query using string interpolation, which can lead to serious security vulnerabilities.\n\n#### Identified Vulnerabilities\n1. **SQL Injection (Critical)**: The use of string interpolation to construct the SQL query makes it vulnerable to SQL injection attacks. An attacker could manipulate the `user_id` input to execute arbitrary SQL commands.\n\n2. **Insecure Dependencies (Medium)**: The function relies on the `db.execute()` method, but without knowing the implementation details of the `db` object, we cannot ascertain if it uses prepared statements or parameterized queries internally. If it does not, the risk of SQL injection remains.\n\n3. **Hardcoded Secrets (Low)**: While there are no hardcoded secrets in the current function, it is important to ensure that any database connection details or crede

The compliance research agent cites regulations (GDPR Articles), flags the jurisdictional differences between Germany and Ireland, and appends the legal disclaimer mandated by its domain layer. The security code review agent structures its output as a severity-rated security report, identifies SQL injection as a critical vulnerability, and proposes parameterized queries - output shapes driven entirely by the behavioral and domain fragments in the composed system prompt. Same model, same underlying capability, completely different behavior from the composition.

## Autonomy layer and LangGraph control flow
The autonomy layer does more than change the system prompt - it changes the **topology of the agent graph**. An `approval_gated` agent needs a human approval node wired into the graph between analysis and execution. A `fully_automated` agent bypasses it entirely. LangGraph's conditional edges make this a first-class architectural choice: the edge routing function reads `approval_required` from the resolved config and routes accordingly. The graph structure is different between the two configurations, not just the prompts.

In [14]:
class AgentState(TypedDict):
    """State passed between nodes in the composed agent graph."""
    messages: Annotated[Sequence[BaseMessage], add_messages]
    autonomy_level: str   # Carried as metadata for logging and display
    proposed_action: str  # Summary of what the agent intends to do next
    approved: bool        # Set by the approval node; checked by the execute node


def build_composed_agent(layers: list):
    """Build a LangGraph agent whose graph structure reflects the composed autonomy mode.

    The key structural decision: if the resolved config has approval_required=True,
    the conditional edge routes through the human approval node before execution.
    Otherwise, analysis routes directly to execution, bypassing the approval node.
    """
    # Resolve config and extract the settings that shape the graph structure
    config = compose_layers(layers)
    system_prompt = build_system_prompt(layers)
    requires_approval = config.get("approval_required", False)  # drives the conditional edge
    autonomy_level = config.get("autonomy_level", "unknown")

    def analyze(state: AgentState) -> dict:
        """LLM processes the task and proposes an action."""
        # Prepend composed system prompt to the message history
        msgs = [SystemMessage(content=system_prompt)] + list(state["messages"])
        response = llm.invoke(msgs)
        return {
            "messages": [response],
            # Store a short summary of the proposed action for the approval gate display
            "proposed_action": response.content[:100] + "...",
            "autonomy_level": autonomy_level,
        }

    def request_approval(state: AgentState) -> dict:
        """Human approval gate — simulated as auto-approve in this demo."""
        print(f"  [APPROVAL GATE] Proposed: {state['proposed_action'][:80]}...")
        print(f"  [APPROVAL GATE] Human reviewed and approved. (simulated)")
        return {"approved": True}

    def execute(state: AgentState) -> dict:
        """Finalize — records whether approval was required or bypassed."""
        if requires_approval:
            status = "executed after human approval"
        else:
            status = "executed autonomously (no approval gate in graph)"
        return {"messages": [AIMessage(content=f"Action {status}.")]}

    def route_after_analyze(state: AgentState) -> str:
        """Conditional edge: route to approval gate or directly to execute."""
        # This single condition is where the autonomy layer shapes the graph topology
        return "request_approval" if requires_approval else "execute"

    graph = StateGraph(AgentState)
    graph.add_node("analyze", analyze)
    graph.add_node("request_approval", request_approval)
    graph.add_node("execute", execute)

    graph.add_edge(START, "analyze")
    # The conditional edge reads from composed config — not hardcoded per-graph logic
    graph.add_conditional_edges("analyze", route_after_analyze, {
        "request_approval": "request_approval",
        "execute": "execute",
    })
    graph.add_edge("request_approval", "execute")
    graph.add_edge("execute", END)

    return graph.compile(), autonomy_level

In [15]:
task = HumanMessage(content="Draft a Q4 GDPR compliance summary for the audit committee.")

# --- approval_gated: graph routes through the human approval node ---
print("--- APPROVAL-GATED AUTONOMY ---")
gated_agent, gated_level = build_composed_agent(
    [approval_gated, research_behavioral, compliance_domain]
)
gated_result = gated_agent.invoke({
    "messages": [task],
    "autonomy_level": gated_level,
    "proposed_action": "",
    "approved": False,
})
print(f"Autonomy level : {gated_result['autonomy_level']}")
print()

# --- fully_automated: the approval node is never reached ---
print("--- FULLY AUTOMATED AUTONOMY ---")
auto_agent, auto_level = build_composed_agent(
    [fully_automated, research_behavioral, compliance_domain]
)
auto_result = auto_agent.invoke({
    "messages": [task],
    "autonomy_level": auto_level,
    "proposed_action": "",
    "approved": False,
})
print(f"Autonomy level : {auto_result['autonomy_level']} (approval node never reached)")

--- APPROVAL-GATED AUTONOMY ---
  [APPROVAL GATE] Proposed: To create a Q4 GDPR compliance summary for the audit committee, I will compile r...
  [APPROVAL GATE] Human reviewed and approved. (simulated)
Autonomy level : approval_gated

--- FULLY AUTOMATED AUTONOMY ---
Autonomy level : fully_automated (approval node never reached)


The `[APPROVAL GATE]` lines appear only for the approval-gated run. The fully automated agent's graph routes directly from `analyze` to `execute` via the conditional edge, never touching the approval node - the node exists in the graph definition but is structurally unreachable given the routing function's output. In production, the `request_approval` node would call a real notification system (a Slack webhook, an email, an enterprise approval portal) and block until a human responds before continuing the graph traversal.